In [14]:
!pip install langchain langchain-community langchain-openai langchain-experimental
!pip install faiss-cpu rank_bm25 unstructured tiktoken
!pip install langchain-huggingface sentence-transformers
!pip install langchain-neo4j ragas datasets

In [15]:
!pip install langchain langchain-community langchain-openai langchain-experimental \
             langchain-text-splitters langchain-huggingface langchain-neo4j \
             faiss-cpu rank_bm25 unstructured tiktoken \
             sentence-transformers neo4j ragas datasets -q

#  Upload File recipe_logic.py

In [16]:
from google.colab import files
files.upload()

Saving recipe_logic.py to recipe_logic (1).py


{'recipe_logic (1).py': b"# recipe_logic.py\r\ndef filter_by_ingredient(recipes, ingredient):\r\n    return [r for r in recipes if ingredient in r['ingredients']]\r\n\r\ndef filter_by_cuisine(recipes, cuisine):\r\n    return [r for r in recipes if r['cuisine'] == cuisine]\r\n\r\ndef filter_by_time(recipes, max_minutes):\r\n    return [r for r in recipes if r['time'] <= max_minutes]"}

# Create Recipe Files

In [17]:
with open("recipes.txt", "w") as f:
    f.write("""Recipe: Spaghetti Bolognese
Cuisine: Italian
Time: 45 minutes
Ingredients: pasta, ground beef, tomatoes, garlic, onion, olive oil, salt, pepper
Steps: Brown the beef. Add garlic and onion. Add tomatoes. Simmer 30 min. Serve over pasta.

Recipe: Chicken Tikka Masala
Cuisine: Indian
Time: 40 minutes
Ingredients: chicken, cream, tomatoes, garam masala, garlic, ginger, onion, butter
Steps: Marinate chicken. Grill it. Make sauce with tomatoes and cream. Combine.

Recipe: Chocolate Lava Cake
Cuisine: Dessert
Time: 25 minutes
Ingredients: chocolate, butter, eggs, flour, sugar, vanilla
Steps: Melt chocolate and butter. Mix with eggs and sugar. Add flour. Bake 12 min.""")

with open("recipes.md", "w") as f:
    f.write("""# Recipe Collection
## Spaghetti Bolognese
- **Cuisine:** Italian
- **Time:** 45 minutes
- **Ingredients:** pasta, ground beef, tomatoes, garlic, onion
- **Steps:** Brown beef. Add vegetables. Simmer. Serve over pasta.

## Chicken Tikka Masala
- **Cuisine:** Indian
- **Time:** 40 minutes
- **Ingredients:** chicken, cream, tomatoes, garam masala, garlic
- **Steps:** Marinate chicken. Grill. Make sauce. Combine.

## Chocolate Lava Cake
- **Cuisine:** Dessert
- **Time:** 25 minutes
- **Ingredients:** chocolate, butter, eggs, flour, sugar
- **Steps:** Melt chocolate. Mix ingredients. Bake 12 minutes.""")

with open("recipes.html", "w") as f:
    f.write("""<html><body>
<h1>Spaghetti Bolognese</h1>
<p>Cuisine: Italian. Time: 45 minutes.</p>
<p>Ingredients: pasta, ground beef, tomatoes, garlic, onion, olive oil</p>
<p>Steps: Brown the beef. Add garlic and onion. Simmer 30 minutes.</p>
<h1>Chicken Tikka Masala</h1>
<p>Cuisine: Indian. Time: 40 minutes.</p>
<p>Ingredients: chicken, cream, tomatoes, garam masala, garlic, ginger</p>
<p>Steps: Marinate chicken. Grill it. Make sauce. Combine.</p>
<h1>Chocolate Lava Cake</h1>
<p>Cuisine: Dessert. Time: 25 minutes.</p>
<p>Ingredients: chocolate, butter, eggs, flour, sugar, vanilla</p>
<p>Steps: Melt chocolate and butter. Mix. Bake 12 minutes.</p>
</body></html>""")

print("All 3 recipe files created!")

All 3 recipe files created!


# Load All Documents

In [18]:
from langchain_community.document_loaders import TextLoader, UnstructuredHTMLLoader

txt_docs  = TextLoader("recipes.txt").load()
md_docs   = TextLoader("recipes.md").load()
html_docs = UnstructuredHTMLLoader("recipes.html").load()
code_docs = TextLoader("recipe_logic.py").load()

all_docs = txt_docs + md_docs + html_docs + code_docs
print(f"recipes.txt loaded:     {len(txt_docs)} doc")
print(f"recipes.md loaded:      {len(md_docs)} doc")
print(f"recipes.html loaded:    {len(html_docs)} doc")
print(f"recipe_logic.py loaded: {len(code_docs)} doc")
print(f"Total: {len(all_docs)} documents")

recipes.txt loaded:     1 doc
recipes.md loaded:      1 doc
recipes.html loaded:    1 doc
recipe_logic.py loaded: 1 doc
Total: 4 documents


# Recursive Splitting

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
r_chunks = recursive_splitter.split_documents(all_docs)

print("=" * 50)
print("RECURSIVE CHARACTER SPLITTING")
print("=" * 50)
for i, c in enumerate(r_chunks[:3]):
    print(f"\nChunk {i+1}:\n{c.page_content}\n{'-'*30}")
print(f"\nTotal recursive chunks: {len(r_chunks)}")

RECURSIVE CHARACTER SPLITTING

Chunk 1:
Recipe: Spaghetti Bolognese
Cuisine: Italian
Time: 45 minutes
Ingredients: pasta, ground beef, tomatoes, garlic, onion, olive oil, salt, pepper
Steps: Brown the beef. Add garlic and onion. Add tomatoes. Simmer 30 min. Serve over pasta.
------------------------------

Chunk 2:
Recipe: Chicken Tikka Masala
Cuisine: Indian
Time: 40 minutes
Ingredients: chicken, cream, tomatoes, garam masala, garlic, ginger, onion, butter
Steps: Marinate chicken. Grill it. Make sauce with tomatoes and cream. Combine.
------------------------------

Chunk 3:
Recipe: Chocolate Lava Cake
Cuisine: Dessert
Time: 25 minutes
Ingredients: chocolate, butter, eggs, flour, sugar, vanilla
Steps: Melt chocolate and butter. Mix with eggs and sugar. Add flour. Bake 12 min.
------------------------------

Total recursive chunks: 11


# Fixed Token Splitting

In [20]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(chunk_size=100, chunk_overlap=10)
t_chunks = token_splitter.split_documents(all_docs)

print("=" * 50)
print("TOKEN-BASED SPLITTING")
print("=" * 50)
for i, c in enumerate(t_chunks[:3]):
    print(f"\nChunk {i+1}:\n{c.page_content}\n{'-'*30}")
print(f"\nTotal token chunks: {len(t_chunks)}")

TOKEN-BASED SPLITTING

Chunk 1:
Recipe: Spaghetti Bolognese
Cuisine: Italian
Time: 45 minutes
Ingredients: pasta, ground beef, tomatoes, garlic, onion, olive oil, salt, pepper
Steps: Brown the beef. Add garlic and onion. Add tomatoes. Simmer 30 min. Serve over pasta.

Recipe: Chicken Tikka Masala
Cuisine: Indian
Time: 40 minutes
Ingredients: chicken, cream, tomatoes, garam masala, garlic, ginger,
------------------------------

Chunk 2:
, garam masala, garlic, ginger, onion, butter
Steps: Marinate chicken. Grill it. Make sauce with tomatoes and cream. Combine.

Recipe: Chocolate Lava Cake
Cuisine: Dessert
Time: 25 minutes
Ingredients: chocolate, butter, eggs, flour, sugar, vanilla
Steps: Melt chocolate and butter. Mix with eggs and sugar. Add flour. Bake 12 min.
------------------------------

Chunk 3:
# Recipe Collection
## Spaghetti Bolognese
- **Cuisine:** Italian
- **Time:** 45 minutes
- **Ingredients:** pasta, ground beef, tomatoes, garlic, onion
- **Steps:** Brown beef. Add veget

# Fixed Semantic Splitting

In [21]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
semantic_splitter = SemanticChunker(embeddings)
s_chunks = semantic_splitter.split_documents(all_docs)

print("=" * 50)
print("SEMANTIC SPLITTING")
print("=" * 50)
for i, c in enumerate(s_chunks[:3]):
    print(f"\nChunk {i+1}:\n{c.page_content}\n{'-'*30}")
print(f"\nTotal semantic chunks: {len(s_chunks)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SEMANTIC SPLITTING

Chunk 1:
Recipe: Spaghetti Bolognese
Cuisine: Italian
Time: 45 minutes
Ingredients: pasta, ground beef, tomatoes, garlic, onion, olive oil, salt, pepper
Steps: Brown the beef. Add garlic and onion. Add tomatoes. Simmer 30 min. Serve over pasta. Recipe: Chicken Tikka Masala
Cuisine: Indian
Time: 40 minutes
Ingredients: chicken, cream, tomatoes, garam masala, garlic, ginger, onion, butter
Steps: Marinate chicken. Grill it.
------------------------------

Chunk 2:
Make sauce with tomatoes and cream. Combine. Recipe: Chocolate Lava Cake
Cuisine: Dessert
Time: 25 minutes
Ingredients: chocolate, butter, eggs, flour, sugar, vanilla
Steps: Melt chocolate and butter. Mix with eggs and sugar. Add flour. Bake 12 min.
------------------------------

Chunk 3:
# Recipe Collection
## Spaghetti Bolognese
- **Cuisine:** Italian
- **Time:** 45 minutes
- **Ingredients:** pasta, ground beef, tomatoes, garlic, onion
- **Steps:** Brown beef. Add vegetables. Simmer. Serve over pasta. ## C

In [61]:
print("=" * 60)
print("COMPARISON OF SPLITTING METHODS")
print("=" * 60)

def show_chunks(name, chunks):
    print(f"\n{name}")
    print("-" * 60)
    print(f"Total Chunks: {len(chunks)}")

    for i, c in enumerate(chunks[:2]):
        print(f"\nChunk {i+1} ({len(c.page_content)} chars):")
        print(c.page_content[:200])
        print("-" * 40)

show_chunks("🔵 Recursive", r_chunks)
show_chunks("🟣 Token", t_chunks)
show_chunks("🟢 Semantic", s_chunks)

COMPARISON OF SPLITTING METHODS

🔵 Recursive
------------------------------------------------------------
Total Chunks: 11

Chunk 1 (235 chars):
Recipe: Spaghetti Bolognese
Cuisine: Italian
Time: 45 minutes
Ingredients: pasta, ground beef, tomatoes, garlic, onion, olive oil, salt, pepper
Steps: Brown the beef. Add garlic and onion. Add tomatoe
----------------------------------------

Chunk 2 (224 chars):
Recipe: Chicken Tikka Masala
Cuisine: Indian
Time: 40 minutes
Ingredients: chicken, cream, tomatoes, garam masala, garlic, ginger, onion, butter
Steps: Marinate chicken. Grill it. Make sauce with toma
----------------------------------------

🟣 Token
------------------------------------------------------------
Total Chunks: 8

Chunk 1 (367 chars):
Recipe: Spaghetti Bolognese
Cuisine: Italian
Time: 45 minutes
Ingredients: pasta, ground beef, tomatoes, garlic, onion, olive oil, salt, pepper
Steps: Brown the beef. Add garlic and onion. Add tomatoe
----------------------------------------

In [22]:
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

vectorstore = FAISS.from_documents(r_chunks, embeddings)
vectorstore.save_local("recipe_faiss_index")

bm25 = BM25Retriever.from_documents(r_chunks)
bm25.k = 3

# SCREENSHOT THIS → Figure 2
print(f"FAISS index size:    {vectorstore.index.ntotal} vectors")
print(f"Embedding dimension: {vectorstore.index.d}")
print(f"BM25 ready with      {len(r_chunks)} documents")

FAISS index size:    11 vectors
Embedding dimension: 384
BM25 ready with      11 documents


In [8]:
!pip install langchain-text-splitters

# Install Neo4j Library

In [23]:
!pip install langchain-neo4j neo4j

# Connect to Neo4j

In [24]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url="YOUR_NEO4J_URL",
    username="neo4j",
    password="YOUR_NEO4J_PASSWORD"
)

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


# LLM

In [3]:
!pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.6 MB/s eta 0:00:00


In [28]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key="YOUR_GROQ_API_KEY"
)
print("LLM ready!")

LLM ready!


# Schema

In [31]:
# Refresh schema first
graph.refresh_schema()

# Then print it
print("=== GRAPH SCHEMA ===")
print(graph.get_schema)

=== GRAPH SCHEMA ===
Node properties:
Recipe {cuisine: STRING, name: STRING, time: INTEGER}
Ingredient {name: STRING}
Cuisine {name: STRING}
Relationship properties:

The relationships:
(:Recipe)-[:BELONGS_TO]->(:Cuisine)
(:Recipe)-[:CONTAINS]->(:Ingredient)


# Build GraphCypherQAChain

In [32]:
!pip install langchain-neo4j -q

In [33]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    validate_cypher=True,
    allow_dangerous_requests=True
)

print("GraphCypherQAChain ready!")

GraphCypherQAChain ready!


# Run 5 Sample Queries

In [34]:
# SCREENSHOT ALL OUTPUTS → Figure 4

queries = [
    "What recipes contain chocolate?",
    "Find recipes from Italian cuisine.",
    "Which recipes contain garlic?",
    "Find recipes that take less than 30 minutes.",
    "What ingredients are in Chicken Tikka Masala?"
]

for q in queries:
    print("\n" + "="*50)
    print(f"Question: {q}")
    try:
        result = chain.invoke({"query": q})
        print(f"Answer: {result['result']}")
    except Exception as e:
        print(f"Error: {e}")


Question: What recipes contain chocolate?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Recipe)-[:CONTAINS]->(i:Ingredient {name: 'chocolate'}) RETURN r.name
Full Context:
[{'r.name': 'Chocolate Lava Cake'}]

> Finished chain.
Answer: Chocolate Lava Cake contains chocolate.

Question: Find recipes from Italian cuisine.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Recipe)-[:BELONGS_TO]->(c:Cuisine {name: 'Italian'}) RETURN r
Full Context:
[{'r': {'name': 'Spaghetti Bolognese', 'cuisine': 'Italian', 'time': 45}}, {'r': {'name': 'Spaghetti Bolognese', 'cuisine': 'Italian', 'time': 45}}]

> Finished chain.
Answer: You can try Spaghetti Bolognese, it's an Italian recipe.

Question: Which recipes contain garlic?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Recipe)-[:CONTAINS]->(i:Ingredient {name: 'garlic'}) RETURN r.name
Full Context:
[{'r.name': 'Spaghetti Bolognese'}, {'r.name': 'Chicken Tikka Masala'}]

> Fi

# Build Hybrid RAG Pipeline (Part 5)

In [49]:
!pip install langchain langchain-community -q

In [52]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def hybrid_retrieve(query):
    bm25_results = bm25.invoke(query)
    vector_results = vector_retriever.invoke(query)
    seen = set()
    combined = []
    for doc in bm25_results + vector_results:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            combined.append(doc)
    return combined

print("Hybrid retriever ready!")

Hybrid retriever ready!


# RAG Chain

In [55]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.
Context: {context}
Question: {question}
Answer:
""")

def get_answer(query):
    docs = hybrid_retrieve(query)
    context = "\n\n".join(d.page_content for d in docs)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": query})

print("RAG chain ready!")

RAG chain ready!


# Queries

In [54]:
rag_queries = [
    "How do I make gluten-free banana bread?",
    "What is the process for making Chocolate Lava Cake?",
    "How do I make Chicken Tikka Masala?",
]

for q in rag_queries:
    print("\n" + "="*50)
    print(f"Question: {q}")
    result = get_answer(q)
    print(f"Answer: {result}")


Question: How do I make gluten-free banana bread?
Answer: The provided context does not contain information about making gluten-free banana bread. The recipes mentioned are for Chocolate Lava Cake and Spaghetti Bolognese, and there is no mention of banana bread.

Question: What is the process for making Chocolate Lava Cake?
Answer: The process for making Chocolate Lava Cake is: 
1. Melt chocolate and butter. 
2. Mix with eggs and sugar. 
3. Add flour. 
4. Bake for 12 minutes.

Question: How do I make Chicken Tikka Masala?
Answer: To make Chicken Tikka Masala, follow these steps: 
1. Marinate the chicken. 
2. Grill the chicken. 
3. Make a sauce with tomatoes and cream. 
4. Combine the grilled chicken with the sauce.


#  RAGAS

In [56]:
!pip install ragas datasets -q

# Run RAGAS Evaluation

In [57]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, answer_relevancy
from datasets import Dataset

# 5 QA pairs
questions = [
    "How do I make Spaghetti Bolognese?",
    "What ingredients are in Chicken Tikka Masala?",
    "How long does Chocolate Lava Cake take?",
    "What are the steps to make pasta?",
    "Which recipes use garlic?",
]
ground_truths = [
    "Brown beef, add tomatoes and garlic, simmer 30 minutes, serve over pasta.",
    "Chicken, cream, tomatoes, garam masala, garlic, ginger, onion, butter.",
    "Chocolate Lava Cake takes 25 minutes.",
    "Boil water, add pasta, cook 10 minutes, drain and serve.",
    "Spaghetti Bolognese and Chicken Tikka Masala use garlic.",
]

# Generate answers and contexts
answers, contexts = [], []
for q in questions:
    docs = hybrid_retrieve(q)
    context_texts = [d.page_content for d in docs]
    answer = get_answer(q)
    answers.append(answer)
    contexts.append(context_texts)
    print(f"✅ Processed: {q}")

print("\nAll answers generated!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_10838/1202138841.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, answer_relevancy
/tmp/ipykernel_10838/1202138841.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections 

✅ Processed: How do I make Spaghetti Bolognese?
✅ Processed: What ingredients are in Chicken Tikka Masala?
✅ Processed: How long does Chocolate Lava Cake take?
✅ Processed: What are the steps to make pasta?
✅ Processed: Which recipes use garlic?

All answers generated!


# Build Dataset and Evaluat

In [60]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# Wrap Groq LLM and HuggingFace embeddings for RAGAS
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Tell each metric to use Groq
faithfulness.llm = ragas_llm
context_precision.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings

# Build dataset
dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths,
})

print("Dataset ready!")
print(dataset)

# Evaluate — SCREENSHOT THIS → Figure 6
scores = evaluate(
    dataset,
    metrics=[faithfulness, context_precision, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n=== RAGAS EVALUATION SCORES ===")
print(scores)

/tmp/ipykernel_10838/1745875937.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, answer_relevancy
/tmp/ipykernel_10838/1745875937.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision, answer_relevancy
/tmp/ipykernel_10838/1745875937.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, context_precision, answer

Dataset ready!
Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[5]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[11]: TimeoutError()



=== RAGAS EVALUATION SCORES ===
{'faithfulness': 0.8750, 'context_precision': 0.4833, 'answer_relevancy': 0.7856}
